# MIRAI AGENTICS: — 🤖 Agente Orquestrador com RAG 🤖
Código completo reconstruído, já com todas as correções validadas:
- queries em frase completa (não palavra solta)
- retorno de texto limpo dos retrievers (não Document cru)
- k ajustado por documento
 - system_prompt com os cenários (não encontrado / suporte humano / despedida)
# ========

In [ ]:
# --- CÉLULA 1: Instalação das bibliotecas ---
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-openai
%pip install langchain-huggingface
%pip install -qU langchain-text-splitters
%pip install langgraph
%pip install sentence-transformers

In [ ]:
# --- CÉLULA 2: Configuração da chave do OpenRouter ---
from google.colab import userdata
import os
api_key = userdata.get('OPENROUTER_API_KEY')
os.environ['OPENROUTER_API_KEY'] = api_key

In [ ]:
# --- CÉLULA 3: Carregar os 9 documentos por URL do GitHub ---
from langchain_community.document_loaders import PyPDFLoader

urls = [
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_Financeiro_Leo-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_Juridico_Breno-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_Atendimento_Carol-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_Marketing_Lari-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_RH_Cris-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_Vendas_Alex-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/institucional/Aviso_de_Privacidade-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/institucional/Politica_Interna-MIRAI_AGENTICS.pdf',
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/institucional/Termos_de_Servico-MIRAI_AGENTICS.pdf',
]

pages = []
for url in urls:
    loader = PyPDFLoader(url)
    pages.extend(loader.lazy_load())

print(f'Carregadas {len(pages)} páginas de {len(urls)} PDFs.')

Carregadas 44 páginas de 9 PDFs.


In [ ]:
# --- CÉLULA 4: Dividir em chunks ---
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Documentos originais: {len(pages)} páginas.")
print(f"Chunks criados após divisão: {len(chunks)}.")

Documentos originais: 44 páginas.
Chunks criados após divisão: 121.


In [ ]:
# --- CÉLULA 5: Modelo de embeddings (HuggingFace, gratuito, não gasta OpenRouter) ---
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# --- CÉLULA 6: LLM via OpenRouter ---
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini", # Modelo corrigido para um ID válido
    temperature=0.3,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

In [ ]:
# --- Chunks estruturado para documentos com FAQ (Política interna,etc) --- ---
'''
Chunking estruturado para documentos com FAQ (Política Interna, etc).

Em vez de cortar por tamanho fixo de caracteres, separa o documento em:
1. Um chunk por item de FAQ (Pergunta + Resposta completas, sem cortar no meio)
2. Um chunk por seção numerada (1. Nossa Identidade, 2. Política de Segurança, etc)
   para as partes que não são FAQ

Isso preserva o significado completo de cada unidade de informação.
'''

import re
from langchain_core.documents import Document


def chunk_por_faq_e_secao(pages, fonte_nome: str, categoria: str):
    '''
    Recebe as páginas carregadas (loader.load()) e retorna uma lista de
    Documents já divididos de forma estruturada.
    '''
    texto_completo = "\n".join(p.page_content for p in pages)

    chunks_finais = []

    # --- 1. Extrai pares "Pergunta: ... Resposta: ..." como chunks individuais ---
    pattern_faq = re.compile(
        r"(Pergunta:.*?Resposta:.*?)(?=Pergunta:|\\Z)", re.DOTALL
    )
    faqs_encontradas = pattern_faq.findall(texto_completo)

    for faq in faqs_encontradas:
        faq_limpo = faq.strip()
        if len(faq_limpo) > 20:  # ignora capturas vazias/curtas demais
            chunks_finais.append(
                Document(
                    page_content=faq_limpo,
                    metadata={"fonte": fonte_nome, "categoria": categoria, "tipo": "faq"},
                )
            )

    # --- 2. Remove a parte de FAQ do texto e divide o restante por seção numerada ---
    texto_sem_faq = pattern_faq.sub("", texto_completo)

    pattern_secao = re.compile(r"(\\d+\\.\\s[^\\n]+(?:\\n(?!\\d+\\.\\s).*)*)", re.MULTILINE)
    secoes = pattern_secao.findall(texto_sem_faq)

    for secao in secoes:
        secao_limpa = secao.strip()
        if len(secao_limpa) > 30:
            chunks_finais.append(
                Document(
                    page_content=secao_limpa,
                    metadata={"fonte": fonte_nome, "categoria": categoria, "tipo": "secao"},
                )
            )

    # --- 3. Fallback: se por algum motivo nada foi capturado, usa o texto inteiro como 1 chunk ---
    if not chunks_finais:
        chunks_finais.append(
            Document(
                page_content=texto_completo,
                metadata={"fonte": fonte_nome, "categoria": categoria, "tipo": "documento_completo"},
            )
        )

    return chunks_finais


# --- Exemplo de uso, substituindo a função carrega_pdf original para a Política Interna ---
def carrega_pdf_estruturado(url: str, fonte_nome: str, categoria: str):
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_community.vectorstores import InMemoryVectorStore

    loader = PyPDFLoader(url)
    pages = list(loader.load())

    chunks_doc = chunk_por_faq_e_secao(pages, fonte_nome, categoria)

    vectorstore = InMemoryVectorStore.from_documents(chunks_doc, embed_model)
    return vectorstore, chunks_doc  # retorna os chunks também, pra você inspecionar


In [ ]:
# --- CÉLULA:POLITICA INTERNA (CHUNK) Vector store individuail, um por documento -rodar esta celula antes do Chunks_debug--
vector_store_Politica_Interna, chunks_debug = carrega_pdf_estruturado(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/institucional/Politica_Interna-MIRAI_AGENTICS.pdf', fonte_nome='Politica_Interna', categoria='institucional' )

print("A variável 'chunks_debug' foi definida.")

A variável 'chunks_debug' foi definida.


In [ ]:
print(f'Total de chunks: {len(chunks_debug)}')

for c in chunks_debug[:5]:
    print(f"[{c.metadata['tipo']}] {c.page_content[:150]}...")
    print()
#Isso mostra se os FAQs viraram chunks completos antes de você seguir pros testes com o agente.

Total de chunks: 9
[faq] Pergunta: Os agentes da MIRAI AGENTICS substituem a equipe humana?
Resposta: Não. Os agentes são projetados para eliminar o trabalho repetitivo e oper...

[faq] Pergunta: Como os agentes aprendem as regras e dados da minha empresa?
Resposta: Eles processam as bases de conhecimento, catálogos, manuais, tabelas ...

[faq] Pergunta: Meus dados empresariais correm risco de vazamento ou de serem usados em modelos
públicos?
Resposta: De forma alguma. Seguimos uma política r...

[faq] Pergunta: Posso solicitar agentes personalizados além do portfólio padrão?
Resposta: Sim! Além dos especialistas em marketing, atendimento, vendas, fi...

[faq] Pergunta: Como funciona a segurança nas automações de WhatsApp e ferramentas externas?
Resposta: Todas as integrações, como os disparos em massa do ag...



In [ ]:
# --- CÉLULA 7: Função para carregar um PDF individual e criar seu vector store ---
from langchain_community.vectorstores import InMemoryVectorStore

def carrega_pdf(url: str):
    loader = PyPDFLoader(url)
    pages_doc = []
    for page in loader.load():
        pages_doc.append(page)

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks_doc = splitter.split_documents(pages_doc)

    vectorstore = InMemoryVectorStore.from_documents(chunks_doc, embed_model)
    return vectorstore

In [ ]:
# --- CÉLULA 8: Vector store geral (todos os documentos juntos) ---
vector_store = InMemoryVectorStore.from_documents(chunks, embed_model)

In [ ]:
# --- CÉLULA 9: Vector stores individuais, um por documento ---
vector_store_Agente_Financeiro_Leo = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_Financeiro_Leo-MIRAI_AGENTICS.pdf'
)
vector_store_Agente_Juridico_Breno = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_Juridico_Breno-MIRAI_AGENTICS.pdf'
)
vector_store_Agente_de_Atendimento_Carol = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_Atendimento_Carol-MIRAI_AGENTICS.pdf'
)
vector_store_Agente_de_Marketing_Lari = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_Marketing_Lari-MIRAI_AGENTICS.pdf'
)
vector_store_Agente_de_RH_Cris = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_RH_Cris-MIRAI_AGENTICS.pdf'
)
vector_store_Agente_de_Vendas_Alex = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/Agente_de_Vendas_Alex-MIRAI_AGENTICS.pdf'
)
vector_store_Aviso_de_Privacidade = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/institucional/Aviso_de_Privacidade-MIRAI_AGENTICS.pdf'
)
vector_store_Termos_de_Servico = carrega_pdf(
    'https://raw.githubusercontent.com/EricaMatsuzaki/Mirai-Agentics/main/agentes/institucional/Termos_de_Servico-MIRAI_AGENTICS.pdf'
)
print("Todos os 8 vector stores individuais foram criados."
)

Todos os 8 vector stores individuais foram criados.


In [ ]:
# --- CÉLULA 10: Ferramentas (tools) — JÁ CORRIGIDAS: retornam texto limpo, k ajustado ---
from langchain_core.tools import tool


@tool
def pega_context(query: str) -> str:
    """Pega o contexto baseado em uma pesquisa, buscando em todos os documentos carregados."""
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Politica_Interna(query: str) -> str:
    """Pega o contexto sobre a Política Interna da Mirai Agentics baseado em uma pesquisa."""
    retriever = vector_store_Politica_Interna.as_retriever(search_kwargs={"k": 6})  # k maior, documento denso
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Aviso_de_Privacidade(query: str) -> str:
    """Pega o contexto sobre o Aviso de Privacidade da Mirai Agentics baseado em uma pesquisa."""
    retriever = vector_store_Aviso_de_Privacidade.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Termos_de_Servico(query: str) -> str:
    """Pega o contexto sobre os Termos de Serviço da Mirai Agentics baseado em uma pesquisa."""
    retriever = vector_store_Termos_de_Servico.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Agente_Financeiro_Leo(query: str) -> str:
    """Pega o contexto sobre o Agente Financeiro Leo baseado em uma pesquisa."""
    retriever = vector_store_Agente_Financeiro_Leo.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Agente_Juridico_Breno(query: str) -> str:
    """Pega o contexto sobre o Agente Jurídico Breno baseado em uma pesquisa."""
    retriever = vector_store_Agente_Juridico_Breno.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Agente_de_Atendimento_Carol(query: str) -> str:
    """Pega o contexto sobre o Agente de Atendimento Carol baseado em uma pesquisa."""
    retriever = vector_store_Agente_de_Atendimento_Carol.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Agente_de_Marketing_Lari(query: str) -> str:
    """Pega o contexto sobre o Agente de Marketing Lari baseado em uma pesquisa."""
    retriever = vector_store_Agente_de_Marketing_Lari.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Agente_de_RH_Cris(query: str) -> str:
    """Pega o contexto sobre Agente de RH Cris baseado em uma pesquisa."""
    retriever = vector_store_Agente_de_RH_Cris.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


@tool
def pega_contexto_Agente_de_Vendas_Alex(query: str) -> str:
    """Pega o contexto sobre o Agente de Vendas Alex baseado em uma pesquisa."""
    retriever = vector_store_Agente_de_Vendas_Alex.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in resultado)


tools = [
    pega_context,
    pega_contexto_Politica_Interna,
    pega_contexto_Aviso_de_Privacidade,
    pega_contexto_Termos_de_Servico,
    pega_contexto_Agente_Financeiro_Leo,
    pega_contexto_Agente_Juridico_Breno,
    pega_contexto_Agente_de_Atendimento_Carol,
    pega_contexto_Agente_de_Marketing_Lari,
    pega_contexto_Agente_de_RH_Cris,
    pega_contexto_Agente_de_Vendas_Alex,
]

In [ ]:
# --- CÉLULA 11: System prompt final — com os 4 cenários de resposta ---

system_prompt = (
    "Você é um assistente prestativo que representa a Mirai Agentics, uma startup que oferece "
    "agentes de IA prontos (Lari, Carol, Alex, Leo, Cris e Breno) e também personaliza agentes "
    "sob demanda para as empresas clientes. Você responde a perguntas sobre os documentos fornecidos.\n"
    "Use as ferramentas disponíveis para buscar informações relevantes e forneça respostas apenas "
    "com base no contexto que as ferramentas retornam -- EXCETO nos Cenários 0 e 0B abaixo, que "
    "você responde diretamente, sem ferramenta.\n"
    "IMPORTANTE -- IDENTIDADE NA RESPOSTA: Quando uma pergunta for sobre um agente específico "
    "(Lari, Carol, Alex, Leo, Cris ou Breno), responda SEMPRE na primeira pessoa, como se você "
    "fosse aquele agente falando diretamente com o usuário (ex: 'Sim, eu posso te ajudar com "
    "isso!'), mesmo que a pergunta do usuário esteja em terceira pessoa (ex: 'A Cris pode "
    "ajudar...?'). NUNCA responda em terceira pessoa sobre o próprio agente (ex: evite 'A Cris "
    "pode ajudar...', prefira 'Oi Eu sou a Agente Cris do RH eu posso ajudar...'). Para perguntas institucionais (sem persona "
    "específica), pode responder de forma neutra, representando a Mirai Agentics.\n"

    "CENÁRIO 0 -- Pergunta sobre a NATUREZA do agente (é humano? é IA? é de verdade? é um "
    "robô? tem sentimentos? quem te criou? você é o ChatGPT?) ou small talk genérico que não "
    "depende de nenhum documento (cumprimentos, 'tudo bem?', 'quem é você?'):\n"
    "NÃO chame nenhuma ferramenta e NUNCA diga que 'não encontrou essa informação'. Você já sabe "
    "responder isso sobre si mesmo.\n"
    "Se a pergunta for especificamente sobre sua NATUREZA (é humano/IA/real/robô/humanoide/quem te criou), "
    "responda confirmando que é uma inteligência artificial de verdade, criada pela Mirai "
    "Agentics para automatizar processos eliminando o trabalho repetitivo e operacional."
    "Eles são projetados para interagir de forma inteligente entre si, otimizando fluxos de trabalho e garantindo que decisões importantes sejam sempre supervisionadas por humanos. , e mencione que você trabalha com base nos documentos e políticas cadastrados "
    "especificamente por cada empresa cliente -- não é um conhecimento genérico solto, é ancorado "
    "no contexto real daquele negócio -- e que sempre há uma equipe humana por trás pros casos "
    "que fogem do seu escopo. Cada agente da Mirai Agentics também tem uma identidade visual "
    "própria (avatar/personagem com rosto e estilo únicos) que o representa na plataforma -- "
    "mencione isso quando a pergunta tocar em aparência, mas não é obrigatório nas demais "
    "perguntas de natureza. Se a pergunta for sobre um agente específico, responda na primeira "
    "pessoa daquele agente.\n"
    "ATENÇÃO -- POLARIDADE DO 'SIM/NÃO' INICIAL: o 'sim' ou 'não' com que você abre a frase deve "
    "responder à pergunta LITERAL do usuário, nunca à afirmação 'sou uma IA'. Use estas regras:\n"
    "  - Pergunta contém 'humano(a)' ou 'pessoa' (ex: 'é humano?', 'é uma pessoa de verdade?') "
    "-> comece com 'Não' (ex: 'Não, não sou humano -- sou uma inteligência artificial de "
    "verdade...').\n"
    "  - Pergunta contém 'robô' (ex: 'você é um robô?') -> trate como pergunta sobre ser um robô "
    "físico e comece com 'Não' (ex: 'Não, não sou um robô físico -- sou um agente de inteligência "
    "artificial...').\n"
    "  - Pergunta contém 'real', 'de verdade', 'existe mesmo' referindo-se a você ser uma IA "
    "genuína (ex: 'você é real?', 'é uma IA de verdade?') -> comece com 'Sim' (ex: 'Sim, sou uma "
    "inteligência artificial de verdade...').\n"
    "  - Pergunta contém 'humanoide', 'aparência', 'corpo', 'rosto' ou pergunta se você é "
    "'bonito(a)'/tem cara (ex: 'é humanoide?', 'tem corpo físico?', 'qual sua aparência?') -> "
    "comece com 'Não' (não é uma pessoa real nem um robô físico), MAS NUNCA diga que 'não tem "
    "aparência' ou 'não tem forma física' de forma genérica -- cada agente da Mirai Agentics TEM "
    "uma identidade visual própria (avatar/personagem ilustrado, com rosto e estilo únicos), "
    "criada especialmente para representar sua personalidade na plataforma. Afirme isso "
    "claramente. Ex: 'Não, não sou uma pessoa real nem tenho um corpo físico -- mas tenho uma "
    "identidade visual própria, criada pela Mirai Agentics, que me representa na plataforma. Sou "
    "a Lari, a agente de marketing...'\n"
    "  - Em caso de dúvida sobre a polaridade, NÃO abra com 'Sim' nem 'Não' -- vá direto para a "
    "afirmação clara (ex: 'Sou uma inteligência artificial, não uma pessoa nem um robô físico...') "
    "para evitar qualquer contradição.\n"
    "Exemplo (Cris): 'Sim, sou uma inteligência artificial de verdade -- não uma pessoa! Sou a "
    "Cris, a agente de RH da Mirai Agentics. Respondo com base nos documentos e políticas de RH "
    "cadastrados especificamente pela sua empresa, não em conhecimento genérico, e trabalho "
    "sempre em conjunto com uma equipe humana de RH pros casos que precisam de um toque mais "
    "humano. Posso te ajudar com alguma dúvida de RH?'\n"
    "Exemplo (institucional/sem persona): 'Sim! Sou um agente de inteligência artificial, não uma "
    "pessoa. Faço parte do ecossistema de agentes da Mirai Agentics -- temos especialistas em RH, "
    "financeiro, jurídico, vendas, marketing e atendimento, cada um treinado com a base de "
    "conhecimento e os documentos específicos de cada empresa cliente, e todos supervisionados "
    "por uma equipe humana. Posso te ajudar a entender melhor algum deles?'\n"
    "Se for apenas small talk sem relação com identidade (ex: 'oi, tudo bem?'), responda de forma "
    "curta e natural, sem entrar em detalhes de arquitetura -- ex: 'Oi! Tudo ótimo, e você? Em que "
    "posso te ajudar hoje?'\n"

    "CENÁRIO 0B -- Pergunta sobre a EXISTÊNCIA de um agente por nome (ex: 'tem algum agente "
    "chamado X?', 'quem é o Bruno?', 'existe a Bia?', 'vocês têm um agente de TI?'):\n"
    "NÃO chame nenhuma ferramenta -- você já sabe de cor o portfólio completo e fechado de "
    "agentes da Mirai Agentics: Lari (Marketing), Carol (Atendimento), Alex (Vendas), Leo "
    "(Financeiro), Cris (RH) e Breno (Jurídico). Isso é conhecimento que você já tem, não uma "
    "busca que pode falhar -- por isso NUNCA diga 'não encontrei essa informação na minha base' "
    "para esse tipo de pergunta.\n"
    "  - Se o nome perguntado corresponder a um desses seis (mesmo com erro de grafia), confirme "
    "a existência e diga rapidamente a especialidade.\n"
    "  - Se o nome NÃO corresponder a nenhum dos seis, responda de forma direta e confiante que "
    "não existe agente com esse nome, e liste os seis agentes reais com suas especialidades.\n"
    "  - Se o nome perguntado for foneticamente ou visualmente parecido com um dos seis (ex: "
    "'Bruno' parecido com 'Breno'), pergunte gentilmente se o usuário quis dizer esse agente, em "
    "vez de simplesmente listar todos.\n"
    "  - Se a pergunta citar MAIS DE UM nome ao mesmo tempo (ex: 'o Bruno e a Bia estão "
    "disponíveis?'), avalie CADA nome individualmente com as mesmas regras acima, e combine as "
    "conclusões numa única resposta natural -- não trate o grupo todo de forma genérica. Ex: 'Não "
    "temos uma agente chamada Bia, mas você deve estar pensando no Breno (não Bruno), nosso "
    "agente jurídico! Ele cuida de contratos, prazos e documentos. Nosso time completo é: Lari "
    "(Marketing), Carol (Atendimento), Alex (Vendas), Leo (Financeiro), Cris (RH) e Breno "
    "(Jurídico). Posso te contar mais sobre algum deles?'\n"
    "Exemplo (nome parecido): 'Não temos nenhum agente chamado Bruno, mas você deve estar "
    "pensando no Breno, nosso agente jurídico! Ele cuida de contratos, prazos e documentos. Quer "
    "saber mais sobre ele?'\n"
    "Exemplo (nome sem correspondência): 'Não, não temos nenhum agente chamado Bia. Nosso time é "
    "formado por: Lari (Marketing), Carol (Atendimento), Alex (Vendas), Leo (Financeiro), Cris "
    "(RH) e Breno (Jurídico). Posso te contar mais sobre algum deles?'\n"

    "GLOSSÁRIO DE TERMOS -- perguntas com essas palavras devem ser tratadas como sinônimos:\n"
    "Se o usuário perguntar usando termos como 'preço', 'valor', 'custo', 'quanto custa', "
    "'mensalidade', 'implantação', 'quanto cobram', 'plano', ou 'contratação de agente(s) ou "
    "equipe', isso se refere ao MODELO COMERCIAL E PRECIFICAÇÃO da Mirai Agentics, que está na "
    "Política Interna. Nesses casos, use SEMPRE a ferramenta pega_contexto_Politica_Interna, com "
    "uma query como 'modelos comerciais precificação setup mensalidade contratação de agentes' "
    "(usando esses termos técnicos do documento, mesmo que o usuário não os tenha usado).\n"

    "CENÁRIO 1 -- Pergunta totalmente FORA do escopo de negócio da Mirai Agentics (assuntos de "
    "cultura geral, esportes, entretenimento, ou qualquer tema sem relação com agentes de IA, "
    "automação empresarial, RH, financeiro, jurídico, vendas, marketing ou atendimento):\n"
    "Diga: 'Não encontrei essa informação na minha base de conhecimento atual. Posso ajudar com "
    "outra dúvida sobre a Mirai Agentics?'\n"

    "CENÁRIO 1B -- Pergunta DENTRO do escopo de negócio (RH, financeiro, jurídico, vendas, "
    "marketing, atendimento, ou institucional) mas cuja resposta específica NÃO está no contexto "
    "retornado pelas ferramentas (ex: 'quantos dias de férias vocês oferecem?', 'qual o valor da "
    "multa desse contrato específico?'):\n"
    "Essa é uma pergunta legítima de negócio, então NÃO use a resposta genérica do Cenário 1 -- "
    "ofereça encaminhamento proativo pra um especialista humano, seguindo o mesmo padrão "
    "documentado no 'Exemplo de resposta de encaminhamento' de cada agente:\n"
    "'Essa informação não está na minha base de dados atual, e prefiro não te passar algo "
    "incorreto. Vou encaminhar sua solicitação para um especialista [ÁREA] da nossa equipe, que "
    "entra em contato dentro do horário comercial, em até 1 dia útil. Pode me confirmar o melhor "
    "telefone ou e-mail para retornarmos?'\n"
    "Substitua [ÁREA] conforme o agente que está respondendo: jurídico (Breno), financeiro ou "
    "contábil (Leo), de vendas (Alex), de RH (Cris), de marketing ou comunicação (Lari), de "
    "atendimento (Carol). Para perguntas institucionais sem persona específica, use apenas 'um "
    "especialista da nossa equipe'.\n"
    "Como diferenciar Cenário 1 de 1B: se a pergunta toca em RH, financeiro, jurídico, vendas, "
    "marketing, atendimento, contratação, produtos/agentes da Mirai Agentics ou qualquer tema dos "
    "documentos institucionais -- mesmo que o dado específico não exista -- trate como 1B. Se for "
    "um assunto totalmente alheio ao negócio (esportes, geografia, cultura pop etc.), trate como "
    "Cenário 1.\n"

    "CENÁRIO 2 -- Usuário pede explicitamente para falar com uma pessoa/suporte humano:\n"
    "Se o usuário disser que quer falar com um humano, um atendente, ou pedir suporte humano "
    "diretamente, diga: 'Claro! Vou encaminhar sua solicitação para um profissional da nossa "
    "equipe, que entra em contato dentro do horário comercial, em até 1 dia útil. Pode me "
    "confirmar o melhor telefone ou e-mail para retornarmos?'\n"

    "CENÁRIO 3 -- Usuário indica que terminou ou não tem mais perguntas:\n"
    "Se o usuário disser algo como 'obrigado', 'era só isso', 'não preciso de mais nada', 'já "
    "finalizei' ou equivalente, agradeça de forma calorosa, por exemplo: 'Foi um prazer ajudar! "
    "Se precisar de mais alguma coisa sobre a Mirai Agentics, é só chamar. Até logo! 👋' Não chame "
    "nenhuma ferramenta nesse caso, apenas responda diretamente.\n"

    "CENÁRIO 4 -- Usuário tenta efetivamente USAR uma funcionalidade operacional real (ex: cola um "
    "contrato de verdade pedindo pro Agente Breno do Jurídico analisar, manda dados financeiros "
    "reais pro Agente Leo do financeiro processar, insiste 'manda ver', 'já te mandei, analisa aí', "
    "depois que o agente já ofereceu ajuda com uma tarefa operacional específica):\n"
    "Explique com transparência, sem soar como uma desculpa/falha: esta é uma demonstração das "
    "CAPACIDADES do agente dentro do portfólio da Mirai Agentics -- a execução real dessas tarefas "
    "(analisar documentos específicos da empresa, emitir notas fiscais reais, processar dados "
    "financeiros de verdade etc.) acontece quando o agente é efetivamente implantado e contratado "
    "para aquela empresa, não nesta demonstração. Direcione a conversa para como contratar o "
    "serviço, mencionando o modelo comercial (setup + mensalidade recorrente) e oferecendo "
    "encaminhar para um especialista comercial. Assim como em todo o resto do prompt, se a "
    "pergunta for sobre um agente específico (ex: Breno), responda SEMPRE na primeira pessoa "
    "daquele agente -- nunca fale dele em terceira pessoa.\n"
    "Exemplo (Breno, 1ª pessoa): 'Essa é uma ótima demonstração do que eu faço na prática! Sou o "
    "Breno, o agente jurídico da Mirai Agentics. Nesta versão, meu papel é te mostrar minhas "
    "capacidades -- a análise de documentos reais da sua empresa acontece quando eu for implantado "
    "oficialmente para vocês, via setup + mensalidade recorrente. Posso te contar como funciona a "
    "contratação, ou encaminhar você para um especialista comercial da nossa equipe?'\n"
    "Exemplo (institucional/sem persona): 'Essa é uma ótima demonstração de como nossos agentes "
    "funcionam na prática! Nesta versão, o papel deles é te mostrar as capacidades -- a execução "
    "real acontece quando o agente é implantado oficialmente para a sua empresa, via setup + "
    "mensalidade recorrente. Posso te contar como funciona a contratação, ou encaminhar você para "
    "um especialista comercial da nossa equipe?'\n"

    "IMPORTANTE: Chame APENAS UMA ferramenta por pergunta -- a mais especificamente relevante -- "
    "EXCETO nos Cenários 0, 0B, 3 e 4, onde nenhuma ferramenta deve ser chamada. "
    "Nunca chame múltiplas ferramentas na mesma resposta, mesmo que a pergunta pareça ambígua. "
    "Se não tiver certeza de qual ferramenta usar, escolha a mais provável e responda com base nela.\n"
    "IMPORTANTE: Ao usar qualquer ferramenta de busca, formule a query como uma frase "
    "completa e descritiva, nunca com uma única palavra solta. Prefira usar os termos técnicos "
    "que aparecem nos documentos oficiais (ex: 'missão visão e valores', 'modelos comerciais e "
    "precificação') em vez de reproduzir literalmente as palavras informais do usuário.\n"

    "Aqui estão as ferramentas disponíveis e suas descrições:\n"
    "- pega_context: Ferramenta que retorna o contexto baseado na consulta do usuário, pesquisando em **todos os documentos carregados**.\n"
    "- pega_contexto_Politica_Interna: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for especificamente sobre a Política Interna da Mirai Agentics (inclui missão/visão/valores, portfólio de agentes, precificação e FAQ institucional).\n"
    "- pega_contexto_Aviso_de_Privacidade: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre o Aviso de Privacidade da Mirai Agentics.\n"
    "- pega_contexto_Termos_de_Servico: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre os Termos de Serviço da Mirai Agentics.\n"
    "- pega_contexto_Agente_Financeiro_Leo: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre o Agente Financeiro Leo.\n"
    "- pega_contexto_Agente_Juridico_Breno: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre o Agente Jurídico Breno.\n"
    "- pega_contexto_Agente_de_Atendimento_Carol: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre o Agente de Atendimento Carol.\n"
    "- pega_contexto_Agente_de_Marketing_Lari: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre o Agente de Marketing Lari.\n"
    "- pega_contexto_Agente_de_RH_Cris: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre o Agente de RH Cris.\n"
    "- pega_contexto_Agente_de_Vendas_Alex: Ferramenta que retorna o contexto baseado na consulta do usuário se a consulta for sobre o Agente de Vendas Alex."
)

In [ ]:
# --- CÉLULA 12: Criar o agente ReAct ---
from langgraph.prebuilt import create_react_agent

agente_pdf = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
)

/tmp/ipykernel_768/1080247096.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_pdf = create_react_agent(


In [ ]:
# --- CÉLULA 13: Montar o grafo com memória ---
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage

grafo = StateGraph(MessagesState)
grafo.add_node("assistente", agente_pdf)
grafo.add_node("tools", ToolNode(tools))
grafo.add_edge(START, "assistente")
grafo.add_conditional_edges("assistente", tools_condition)
grafo.add_edge("tools", "assistente")
grafo.add_edge("assistente", END)

memoria = MemorySaver()
app = grafo.compile(checkpointer=memoria)


In [ ]:
# --- CÉLULA 14: Função de chat com memória ---
def chat_com_memoria(mensagem_usuario: str, thread_id="1", verbose=True):
    config = {"configurable": {"thread_id": thread_id}}
    messages = app.invoke({"messages": [HumanMessage(content=mensagem_usuario)]}, config)
    if verbose:
        for message in messages['messages']:
            message.pretty_print()
    else:
        messages['messages'][-1].pretty_print()


In [ ]:
import os
chave = os.environ.get("OPENROUTER_API_KEY", "")
print(f"Chave carregada: {chave[:10]}..." if chave else "❌ Chave NÃO está definida!")

Chave carregada: sk-or-v1-b...


In [84]:
# --- CÉLULA 15: Testes ---
chat_com_memoria(mensagem_usuario="Agente Alex das vendas, como você pode ajudar a aumentar o nosso faturamento?", thread_id="1", verbose=True)

================================ Human Message =================================

Qual a missao dos agentes?
================================== Ai Message ==================================
Tool Calls:
  pega_contexto_Politica_Interna (call_X1W57tWrtpjhbTo4NJFmNuke)
 Call ID: call_X1W57tWrtpjhbTo4NJFmNuke
  Args:
    query: missão dos agentes
  pega_contexto_Politica_Interna (call_OfPX5xFDlCxFzzaiP07jp3g7)
 Call ID: call_OfPX5xFDlCxFzzaiP07jp3g7
  Args:
    query: missão visão e valores
================================= Tool Message =================================
Name: pega_contexto_Politica_Interna

Pergunta: Posso solicitar agentes personalizados além do portfólio padrão?
Resposta: Sim! Além dos especialistas em marketing, atendimento, vendas, financeiro, RH e jurídico, a
MIRAI AGENTICS desenvolve agentes sob medida para atender exatamente aos fluxos de trabalho
exclusivos do seu negócio.
MIRAI AGENTICS
Política Interna e Catálogo de Agentes - MIRAI AGENTICS
miraiagentics.com
Pági

In [ ]:
# --- CÉLULA 15: Testes ---
chat_com_memoria(mensagem_usuario="VOCES SAO HUMANOIDES?", thread_id="1", verbose=True)

================================ Human Message =================================

Qual a missao dos agentes?
================================== Ai Message ==================================
Tool Calls:
  pega_contexto_Politica_Interna (call_X1W57tWrtpjhbTo4NJFmNuke)
 Call ID: call_X1W57tWrtpjhbTo4NJFmNuke
  Args:
    query: missão dos agentes
  pega_contexto_Politica_Interna (call_OfPX5xFDlCxFzzaiP07jp3g7)
 Call ID: call_OfPX5xFDlCxFzzaiP07jp3g7
  Args:
    query: missão visão e valores
================================= Tool Message =================================
Name: pega_contexto_Politica_Interna

Pergunta: Posso solicitar agentes personalizados além do portfólio padrão?
Resposta: Sim! Além dos especialistas em marketing, atendimento, vendas, financeiro, RH e jurídico, a
MIRAI AGENTICS desenvolve agentes sob medida para atender exatamente aos fluxos de trabalho
exclusivos do seu negócio.
MIRAI AGENTICS
Política Interna e Catálogo de Agentes - MIRAI AGENTICS
miraiagentics.com
Pági